# KissanSevaAI Phase 1 — Full-Dataset DINOv2-S/14 Embedding & Versioned FAISS Pipeline

**Scope (Task 3 + Task 4 Eval Prep):**
- Extracts 768-D L2-normalized DINOv2-S/14 (`CLS + mean patch`) embeddings from **all eligible training images** in `Plant-Diseases-100k`.
- Preserves the exact **67-class canonical order** matching `ml-service/model_assets/classes.json`.
- Uses the exact preprocessing & model weights required by `dinov2_vits14_cls_mean.onnx`, with automated PyTorch↔ONNX equivalence verification.
- Removes byte-identical (`SHA256`) duplicates and uses **perceptual-hash (`dHash`) grouped stratified 80/10/10 splits** so near-duplicate leaf photos cannot leak across `train`, `val`, and `test`.
- Stores resumable atomic chunks, builds an exact normalized **`faiss.IndexFlatIP(768)`** containing **training vectors only**, and saves **versioned** outputs without overwriting production `v1` assets.


## 1. Environment Setup, Drive Mount & Versioning Guard


In [ ]:
!pip -q install faiss-cpu scikit-learn pandas pillow tqdm onnx onnxruntime pyarrow

import hashlib
import json
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import faiss
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> GPU (T4/L4/A100)'
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Versioned output directory — NEVER overwrites existing pilot/v1 production assets
VERSION_TAG = 'v2_full_dinov2_vits14'
ZIP_PATH = Path('/content/drive/MyDrive/archive.zip')  # Adjust if stored elsewhere in Drive
EXISTING_V1_ASSETS_DIR = Path('/content/drive/MyDrive/KissanSevaAI_Phase1/pilot')  # Optional reference
WORK_DIR = Path('/content/plant_disease_100k')
OUT_DIR = Path('/content/drive/MyDrive/KissanSevaAI_Phase1/versions') / VERSION_TAG
ASSETS_OUT = OUT_DIR / 'model_assets'
EVAL_OUT = OUT_DIR / 'eval_cache'
CHUNKS_DIR = OUT_DIR / 'embedding_chunks'

for p in [OUT_DIR, ASSETS_OUT, EVAL_OUT, CHUNKS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.is_file() and zipfile.is_zipfile(ZIP_PATH), f'Missing valid archive: {ZIP_PATH}'
assert ASSETS_OUT.resolve() != EXISTING_V1_ASSETS_DIR.resolve(), 'Guard: Versioned output must not match v1 dir'

BATCH_SIZE = 32   # Lower to 16 if VRAM constrained
CHUNK_SIZE = 1024 # Atomic Drive checkpoint size
SIZE = 224
RESIZE_SHORT = 256
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
print('Versioned Output:', OUT_DIR)


## 2. Canonical 67-Class Mapping (Preserved 1:1 from Production `classes.json`)


In [ ]:
CANONICAL_CLASSES = [
  "Apple/Apple_Black_rot",
  "Apple/Apple_Cedar_apple_rust",
  "Apple/Apple_healthy",
  "Apple/Apple_scab",
  "Cherry/Cherry_Healthy",
  "Cherry/Cherry_Powdery_mildew",
  "Citrus/Citrus_Black_spot",
  "Citrus/Citrus_Healthy",
  "Citrus/Citrus_canker",
  "Citrus/Citrus_greening",
  "Corn/Corn_Common_rust",
  "Corn/Corn_Northern_Leaf_Blight",
  "Corn/Corn_cercospora_gray_leaf_spot",
  "Corn/Corn_healthy",
  "Cotton/Cotton_bacterial_blight",
  "Cotton/Cotton_curl_virus",
  "Cotton/Cotton_fussarium_wilt",
  "Cotton/Cotton_healthy",
  "Grape/Grape_Black_Measles",
  "Grape/Grape_Black_rot",
  "Grape/Grape_Healthy",
  "Grape/Grape_Isariopsis_Leaf_Spot",
  "Peach/Peach_Bacterial_spot",
  "Peach/Peach_Healthy",
  "Pepper_bell/Pepper_bell_Bacterial_spot",
  "Pepper_bell/pepper_bell_healthy",
  "Potato/Potato_Early_blight",
  "Potato/Potato_Late_blight",
  "Potato/Potato_healthy",
  "Rice/Rice_Bacterial_Leaf_Blight",
  "Rice/Rice_Brown_Spot",
  "Rice/Rice_Healthy",
  "Rice/Rice_Leaf_Blast",
  "Rice/Rice_Leaf_scald",
  "Rice/Rice_Sheath_Blight",
  "Strawberry/Strawberry_Healthy",
  "Strawberry/Strawberry_Leaf_scorch",
  "Sugarcane/Sugarcane_Healthy",
  "Sugarcane/Sugarcane_Mosaic_Virus",
  "Sugarcane/Sugarcane_RedRot",
  "Sugarcane/Sugarcane_Rust",
  "Sugarcane/Sugarcane_Yellow",
  "Tomato/Tomato_Bacterial_spot",
  "Tomato/Tomato_Early_blight",
  "Tomato/Tomato_Late_blight",
  "Tomato/Tomato_Leaf_Mold",
  "Tomato/Tomato_Septoria_leaf_spot",
  "Tomato/Tomato_Spider_mites",
  "Tomato/Tomato_Target_Spot",
  "Tomato/Tomato_YellowLeaf_Curl_Virus",
  "Tomato/Tomato_healthy",
  "Tomato/Tomato_mosaic_virus",
  "Wheat/Wheat_Aphid",
  "Wheat/Wheat_Black_Rust",
  "Wheat/Wheat_Blast",
  "Wheat/Wheat_Brown_Rust",
  "Wheat/Wheat_Common_Root_Rot",
  "Wheat/Wheat_Fusarium_Head_Blight",
  "Wheat/Wheat_Healthy",
  "Wheat/Wheat_Leaf_Blight",
  "Wheat/Wheat_Mildew",
  "Wheat/Wheat_Mite",
  "Wheat/Wheat_Septoria",
  "Wheat/Wheat_Smut",
  "Wheat/Wheat_Stem_fly",
  "Wheat/Wheat_Tan_spot",
  "Wheat/Wheat_Yellow_Rust"
]
assert len(CANONICAL_CLASSES) == 67 and len(set(CANONICAL_CLASSES)) == 67
CLASS_TO_ID = {name: idx for idx, name in enumerate(CANONICAL_CLASSES)}
(ASSETS_OUT / 'classes.json').write_text(json.dumps(CANONICAL_CLASSES, indent=2))
print('Preserved canonical classes:', len(CANONICAL_CLASSES), '| ID range: 0..', len(CANONICAL_CLASSES)-1)


## 3. Extract Dataset Safely & Audit Eligible Training Images


In [ ]:
DONE_FLAG = WORK_DIR / '.extract_done'
if not DONE_FLAG.exists():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        base = WORK_DIR.resolve()
        for member in z.infolist():
            target = (base / member.filename).resolve()
            if not (base == target or base in target.parents):
                raise ValueError(f'Unsafe archive member path: {member.filename}')
        z.extractall(WORK_DIR)
    DONE_FLAG.touch()

crop_roots = [p for p in WORK_DIR.rglob('crops') if p.is_dir()]
assert crop_roots, 'Could not locate crops/ directory inside extracted dataset'
CROPS_DIR = min(crop_roots, key=lambda p: len(p.parts))
print('CROPS_DIR:', CROPS_DIR)


## 4. Deduplicate Exact & Perceptual Near-Duplicates, Enforce Canonical 67 Classes, Build Leakage-Controlled 80/10/10 Split

Notice: We filter strictly to `CANONICAL_CLASSES` (the 67 supported labels), reject unreadable/corrupt/tiny images (`< 32px`), remove all byte-identical `SHA256` duplicates (and conflicting labels), and group images by `(label, dHash)` via `StratifiedGroupKFold` so near-identical photos never leak across `train`, `val`, and `test`.


In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
SPLITS_CSV = OUT_DIR / 'splits_v2_audited.csv'

def compute_dhash(img: Image.Image, hash_size: int = 8) -> str:
    gray = ImageOps.exif_transpose(img).convert('L').resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    arr = np.asarray(gray, dtype=np.int16)
    diff = arr[:, 1:] > arr[:, :-1]
    bits = 0
    for b in diff.flatten():
        bits = (bits << 1) | int(bool(b))
    return f'{bits:016x}'

if SPLITS_CSV.exists():
    split_df = pd.read_csv(SPLITS_CSV)
    print('Loaded existing audited splits:', len(split_df))
else:
    candidates = []
    skipped_unmapped = set()
    for crop_dir in sorted(CROPS_DIR.iterdir()):
        if not crop_dir.is_dir():
            continue
        for cond_dir in sorted(crop_dir.iterdir()):
            if not cond_dir.is_dir():
                continue
            label = f'{crop_dir.name}/{cond_dir.name}'
            if label not in CLASS_TO_ID:
                skipped_unmapped.add(label)
                continue
            for img_path in sorted(cond_dir.rglob('*')):
                if img_path.is_file() and img_path.suffix.lower() in EXTS:
                    candidates.append((str(img_path.relative_to(CROPS_DIR)), label, CLASS_TO_ID[label]))

    print(f'Discovered {len(candidates)} candidate files across {len(CLASS_TO_ID)} canonical classes. Skipped unmapped dirs: {len(skipped_unmapped)}')

    seen_sha = {}
    conflict_shas = set()
    records = []
    corrupt_count = 0
    exact_dupe_count = 0

    for rel_path, label, class_id in tqdm(candidates, desc='SHA256 + Image Integrity + dHash'):
        full_p = CROPS_DIR / rel_path
        raw_bytes = full_p.read_bytes()
        sha = hashlib.sha256(raw_bytes).hexdigest()
        if sha in seen_sha:
            if seen_sha[sha] != label:
                conflict_shas.add(sha)
            exact_dupe_count += 1
            continue
        seen_sha[sha] = label
        try:
            with Image.open(full_p) as im:
                im.load()
                w, h = im.size
                if w < 32 or h < 32 or (w * h) > 40_000_000:
                    corrupt_count += 1
                    continue
                dh = compute_dhash(im)
        except Exception:
            corrupt_count += 1
            continue

        records.append({
            'sample_id': len(records),
            'path': rel_path,
            'label': label,
            'crop': label.split('/')[0],
            'class_id': class_id,
            'sha256': sha,
            'dhash': dh,
            'width': w,
            'height': h,
        })

    df = pd.DataFrame(records)
    if conflict_shas:
        df = df[~df.sha256.isin(conflict_shas)].reset_index(drop=True)

    # Leakage-controlled StratifiedGroupKFold using perceptual cluster (label::dhash)
    df['group_key'] = df['label'] + '::' + df['dhash']
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    fold_ids = np.zeros(len(df), dtype=np.int32)
    for fold, (_, test_idx) in enumerate(sgkf.split(df, df['class_id'], df['group_key'])):
        fold_ids[test_idx] = fold

    # Folds 0..7 => train (80%), fold 8 => val (10%), fold 9 => test (10%)
    df['split'] = np.where(fold_ids == 9, 'test', np.where(fold_ids == 8, 'val', 'train'))
    df = df.sort_values(['split', 'class_id', 'path']).reset_index(drop=True)
    df.to_csv(SPLITS_CSV, index=False)
    split_df = df
    print(f'Kept {len(split_df)} unique valid images | Dropped exact dupes={exact_dupe_count}, conflicts={len(conflict_shas)}, corrupt={corrupt_count}')

# Strict Leakage Verification
assert set(split_df['class_id'].unique()) == set(range(67)), 'Missing one or more of the 67 canonical classes!'
train_df = split_df[split_df.split == 'train'].reset_index(drop=True)
val_df = split_df[split_df.split == 'val'].reset_index(drop=True)
test_df = split_df[split_df.split == 'test'].reset_index(drop=True)

assert set(train_df.sha256).isdisjoint(val_df.sha256)
assert set(train_df.sha256).isdisjoint(test_df.sha256)
assert set(val_df.sha256).isdisjoint(test_df.sha256)
assert set(train_df.group_key).isdisjoint(val_df.group_key)
assert set(train_df.group_key).isdisjoint(test_df.group_key)
print('Split Counts:', split_df.split.value_counts().to_dict())
print('Zero SHA256 & perceptual dHash group overlap verified across Train / Val / Test!')


## 5. Load Exact DINOv2-S/14 Weights, Shared Preprocessing & Export Verification


In [ ]:
PREPROCESS = transforms.Compose([
    transforms.Resize(RESIZE_SHORT, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True).eval().to(DEVICE)
backbone.requires_grad_(False)

class DinoFeatures(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, x):
        feat = self.encoder.forward_features(x)
        cls = feat['x_norm_clstoken']
        avg = feat['x_norm_patchtokens'].mean(dim=1)
        return F.normalize(torch.cat([cls, avg], dim=1), dim=1)

model = DinoFeatures(backbone).eval().to(DEVICE)

# Export compatible ONNX into versioned ASSETS_OUT and verify numerical parity
onnx_out = ASSETS_OUT / 'dinov2_vits14_cls_mean.onnx'
dummy = torch.randn(1, 3, SIZE, SIZE, device=DEVICE)
with torch.inference_mode():
    torch.onnx.export(
        model, dummy, str(onnx_out),
        input_names=['image'], output_names=['embedding'],
        dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
        opset_version=17, do_constant_folding=True, dynamo=False
    )
    pt_out = model(dummy).float().cpu().numpy()

ort_sess = ort.InferenceSession(str(onnx_out), providers=['CPUExecutionProvider'])
onx_out = ort_sess.run(None, {'image': dummy.detach().cpu().numpy()})[0]
max_diff = float(np.abs(pt_out - onx_out).max())
print(f'PyTorch <-> ONNX max abs diff: {max_diff:.6f} | L2 norm: {np.linalg.norm(onx_out):.6f}')
assert max_diff < 0.01, f'ONNX mismatch too high: {max_diff}'

# Optional check against existing v1 ONNX file if uploaded to Drive
v1_onnx = EXISTING_V1_ASSETS_DIR / 'dinov2_vits14_cls_mean.onnx'
if v1_onnx.exists():
    v1_sess = ort.InferenceSession(str(v1_onnx), providers=['CPUExecutionProvider'])
    v1_out = v1_sess.run(None, {'image': dummy.detach().cpu().numpy()})[0]
    cos_v1 = float(np.dot(onx_out[0], v1_out[0]))
    print(f'New ONNX <-> v1 ONNX output cosine similarity: {cos_v1:.6f}')
    assert cos_v1 > 0.9999, 'Encoder weights diverge from production v1 ONNX!' 


## 6. Resumable Chunked Embedding Extraction (Training Set Indexed; Val/Test Cached Separately for Eval)

- Every chunk saves deterministically using `.tmp.npy` -> `os.replace()` so Colab disconnects resume cleanly without duplicates or corruption.
- **Only `train` embeddings** enter `crop_index.faiss`. `val` and `test` embeddings are stored solely in `eval_cache/` for Task 4 and NEVER added to the index.


In [ ]:
class IndexedDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        img_path = CROPS_DIR / self.frame.iloc[i]['path']
        with Image.open(img_path) as raw:
            img = ImageOps.exif_transpose(raw).convert('RGB')
            return PREPROCESS(img)

def extract_split_embeddings(part_name: str, part_df: pd.DataFrame):
    assert part_df['sha256'].is_unique, f'Duplicate SHA256 found inside {part_name}!'
    part_chunk_dir = CHUNKS_DIR / part_name
    part_chunk_dir.mkdir(parents=True, exist_ok=True)
    gathered = []

    for start in tqdm(range(0, len(part_df), CHUNK_SIZE), desc=f'Extract [{part_name}]'):
        stop = min(start + CHUNK_SIZE, len(part_df))
        chunk_file = part_chunk_dir / f'chunk_{start:07d}_{stop:07d}.npy'
        meta_file = part_chunk_dir / f'chunk_{start:07d}_{stop:07d}.json'
        expected_shas = part_df.iloc[start:stop]['sha256'].tolist()

        loaded_ok = False
        if chunk_file.exists() and meta_file.exists():
            recorded = json.loads(meta_file.read_text())
            if recorded.get('sha256_list') == expected_shas:
                arr = np.load(chunk_file)
                if arr.shape == (stop - start, 768) and np.isfinite(arr).all():
                    loaded_ok = True

        if not loaded_ok:
            ds = IndexedDataset(part_df.iloc[start:stop])
            dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
            batch_vecs = []
            with torch.inference_mode():
                for xb in dl:
                    with torch.autocast('cuda', dtype=torch.float16):
                        emb = model(xb.to(DEVICE, non_blocking=True)).float().cpu().numpy()
                    batch_vecs.append(emb)
            arr = np.concatenate(batch_vecs, axis=0).astype('float32')
            arr /= np.maximum(np.linalg.norm(arr, axis=1, keepdims=True), 1e-9)
            tmp_npy = chunk_file.with_suffix('.tmp.npy')
            np.save(tmp_npy, arr)
            os.replace(tmp_npy, chunk_file)
            meta_file.write_text(json.dumps({'start': start, 'stop': stop, 'sha256_list': expected_shas}))

        gathered.append(arr)

    X = np.concatenate(gathered, axis=0).astype('float32')
    y = part_df['class_id'].to_numpy(dtype=np.int32)
    assert X.shape == (len(part_df), 768) and np.isfinite(X).all()
    norms = np.linalg.norm(X, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-4), f'L2 norm drift detected: min={norms.min()}, max={norms.max()}'
    return X, y

X_train, y_train = extract_split_embeddings('train', train_df)
X_val, y_val = extract_split_embeddings('val', val_df)
X_test, y_test = extract_split_embeddings('test', test_df)

np.save(EVAL_OUT / 'X_val.npy', X_val)
np.save(EVAL_OUT / 'y_val.npy', y_val)
np.save(EVAL_OUT / 'X_test.npy', X_test)
np.save(EVAL_OUT / 'y_test.npy', y_test)
print('All splits extracted:', X_train.shape, X_val.shape, X_test.shape)


## 7. Build Normalized Exact `faiss.IndexFlatIP` + Save Versioned Metadata & Provenance

- Uses exact **`faiss.IndexFlatIP(768)`** (100% recall, zero IVF quantization loss).
- Saves class IDs (`train_labels.npy`), full source image provenance (`train_image_mapping.csv`), and SHA256 manifest in versioned folder.


In [ ]:
assert len(X_train) == len(train_df) == len(y_train)
assert train_df['sha256'].is_unique

# Strict normalized exact inner-product FAISS index
full_index = faiss.IndexFlatIP(768)
full_index.add(X_train)
assert full_index.ntotal == len(train_df)
assert full_index.metric_type == faiss.METRIC_INNER_PRODUCT

faiss_out = ASSETS_OUT / 'crop_index.faiss'
faiss.write_index(full_index, str(faiss_out))
np.save(ASSETS_OUT / 'train_labels.npy', y_train)

train_meta_df = train_df[['sample_id', 'path', 'label', 'crop', 'class_id', 'sha256', 'dhash', 'width', 'height']].copy()
train_meta_df.insert(0, 'faiss_row_id', np.arange(len(train_meta_df)))
train_meta_df.to_csv(ASSETS_OUT / 'train_image_mapping.csv', index=False)

# Post-index near-duplicate embedding leakage check (Val -> Train)
val_top1_sims, _ = full_index.search(X_val, 1)
near_dup_val = int((val_top1_sims[:, 0] >= 0.98).sum())
print(f'FAISS IndexFlatIP built: {full_index.ntotal} vectors | Val->Train cos>=0.98 near-dupes: {near_dup_val}/{len(X_val)}')

meta_v2 = {
    'asset_version': VERSION_TAG,
    'model': 'dinov2_vits14',
    'features': 'normalized_concat_cls_and_mean_patch',
    'embedding_dim': 768,
    'image_size': SIZE,
    'resize_short_side': RESIZE_SHORT,
    'mean': MEAN,
    'std': STD,
    'class_count': 67,
    'k': 1,
    'vote_method': 'majority',
    'cosine_metric': 'IndexFlatIP',
    'nprobe': None,
    'train_reference_count': int(full_index.ntotal),
    'val_count': int(len(X_val)),
    'test_count': int(len(X_test)),
    'val_near_dupes_ge_0_98': near_dup_val,
    'note': 'Versioned full-dataset index. Pending Task 4 validation tuning before promotion.',
}
(ASSETS_OUT / 'model_meta.json').write_text(json.dumps(meta_v2, indent=2))

# Build SHA256 manifest for integrity verification during local import
manifest = {}
for fname in ['classes.json', 'crop_index.faiss', 'dinov2_vits14_cls_mean.onnx', 'model_meta.json', 'train_labels.npy', 'train_image_mapping.csv']:
    fpath = ASSETS_OUT / fname
    manifest[fname] = {
        'size_bytes': fpath.stat().st_size,
        'sha256': hashlib.sha256(fpath.read_bytes()).hexdigest(),
    }
(ASSETS_OUT / 'SHA256_MANIFEST.json').write_text(json.dumps(manifest, indent=2))

print('=== Task 3 Versioned Assets Ready ===')
for fname, info in manifest.items():
    mb = info['size_bytes'] / (1024 ** 2)
    h12 = info['sha256'][:12]
    print(f'  {fname:30s} | {mb:6.2f} MB | sha256={h12}...')


## 8. (Task 4 Scaffold — Disabled Until Explicitly Approved)

Set `RUN_TASK4_VALIDATION = True` once approved to benchmark on **`val` ONLY**:
- `k in [1, 3, 5, 7, 11, 21]` (Majority vs Similarity-Weighted)
- Global retrieval vs Crop-Aware retrieval
- Nearest Class Prototypes
- Logistic Regression linear head & SVM

Keep `RUN_FINAL_TEST_ONCE = False` until a single winner is chosen on `val`.


In [ ]:
RUN_TASK4_VALIDATION = False  # Approval gate before GPU/experiment sweeps
RUN_FINAL_TEST_ONCE = False   # Unlock strictly once for final winning config

def predict_knn(sims, ids, y_ref, k, method='weighted', allowed_mask=None):
    preds, agrees = [], []
    for ss, ii in zip(sims[:, :k], ids[:, :k]):
        valid = ii >= 0
        nbr_y = y_ref[ii[valid]]
        nbr_s = ss[valid]
        w = np.ones(len(nbr_y)) if method == 'majority' else np.exp(10.0 * (nbr_s - nbr_s.max()))
        tallies = np.bincount(nbr_y, weights=w, minlength=67)
        if allowed_mask is not None:
            tallies = tallies * allowed_mask
        p = int(np.argmax(tallies))
        preds.append(p)
        agrees.append(float(tallies[p] / max(tallies.sum(), 1e-8)))
    return np.array(preds), np.array(agrees)

if RUN_TASK4_VALIDATION:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import LinearSVC

    k_max = 21
    val_sims, val_ids = full_index.search(X_val, k_max)
    exp_rows = []
    for k in [1, 3, 5, 7, 11, 21]:
        for method in (['majority'] if k == 1 else ['majority', 'weighted']):
            t0 = time.perf_counter()
            vp, _ = predict_knn(val_sims, val_ids, y_train, k, method)
            ms = (time.perf_counter() - t0) * 1000.0 / len(X_val)
            exp_rows.append({
                'method': f'global_knn_k{k}_{method}',
                'val_accuracy': accuracy_score(y_val, vp),
                'val_macro_f1': f1_score(y_val, vp, average='macro', zero_division=0),
                'ms_per_query': ms,
            })

    # Crop-Aware Oracle Upper-Bound Simulation (filter neighbors to ground-truth crop family)
    crop_of_cls = np.array([c.split('/')[0] for c in CANONICAL_CLASSES])
    for k in [1, 5, 11]:
        preds_ca = []
        for idx_q in range(len(X_val)):
            true_crop = crop_of_cls[y_val[idx_q]]
            mask = (crop_of_cls == true_crop).astype(np.float64)
            ss = val_sims[idx_q, :k]
            ii = val_ids[idx_q, :k]
            valid = ii >= 0
            w = np.exp(10.0 * (ss[valid] - ss[valid].max()))
            tallies = np.bincount(y_train[ii[valid]], weights=w, minlength=67) * mask
            preds_ca.append(int(np.argmax(tallies)))
        exp_rows.append({
            'method': f'crop_aware_knn_k{k}_weighted',
            'val_accuracy': accuracy_score(y_val, preds_ca),
            'val_macro_f1': f1_score(y_val, preds_ca, average='macro', zero_division=0),
            'ms_per_query': 0.1,
        })

    centroids = np.vstack([X_train[y_train == c].mean(axis=0) for c in range(67)]).astype('float32')
    faiss.normalize_L2(centroids)
    c_pred = np.argmax(X_val @ centroids.T, axis=1)
    exp_rows.append({
        'method': 'class_prototypes_cosine',
        'val_accuracy': accuracy_score(y_val, c_pred),
        'val_macro_f1': f1_score(y_val, c_pred, average='macro', zero_division=0),
        'ms_per_query': 0.01,
    })

    lr = LogisticRegression(max_iter=1000, C=5.0, n_jobs=-1).fit(X_train, y_train)
    lr_pred = lr.predict(X_val)
    exp_rows.append({
        'method': 'linear_head_logreg',
        'val_accuracy': accuracy_score(y_val, lr_pred),
        'val_macro_f1': f1_score(y_val, lr_pred, average='macro', zero_division=0),
        'ms_per_query': 0.05,
    })

    svm = LinearSVC(C=2.0).fit(X_train, y_train)
    svm_pred = svm.predict(X_val)
    exp_rows.append({
        'method': 'linear_svc',
        'val_accuracy': accuracy_score(y_val, svm_pred),
        'val_macro_f1': f1_score(y_val, svm_pred, average='macro', zero_division=0),
        'ms_per_query': 0.05,
    })

    res_df = pd.DataFrame(exp_rows).sort_values(['val_macro_f1', 'val_accuracy'], ascending=False)
    print(res_df.to_string(index=False))
    res_df.to_csv(OUT_DIR / 'task4_validation_comparison.csv', index=False)
